# Session 2: compound growth and the EMI
**A. Real-world problem.** Most loans are not paid back in one lump. You pay the same
amount every month, the EMI (equated monthly instalment). Where does that number come
from, how is each payment split between interest and principal, and why does a longer
loan cost so much more in total?

Convention used throughout: annual rate `annual` (0.12 = 12 percent a year), monthly
rate r = annual / 12, `n` months.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: compound growth, n times a year, and the continuous limit
Adding interest n times a year:  A = P (1 + r/n)^(n t).  Let n grow: the amount creeps up
towards the limit  P e^(r t)  (continuous compounding) and never passes it.

In [ ]:
def compound_growth(P, rate, n, t):
    """A = P (1 + rate/n)^(n t): interest added n times a year for t years."""
    return P * (1.0 + rate / n) ** (n * t)


def continuous_growth(P, rate, t):
    return P * float(np.exp(rate * t))

## Step 2: the EMI formula
Borrow P, pay the same EMI at the end of each of n months, monthly rate r:
    EMI = P r (1+r)^n / ((1+r)^n - 1).
(If r = 0 it is simply P / n.)

In [ ]:
def emi(P, annual, months):
    r = annual / 12.0
    if r == 0:
        return P / months
    g = (1.0 + r) ** months
    return P * r * g / (g - 1.0)

## Step 3: check the formula by simulating the balance
Each month: interest = balance * r is added, then you pay the EMI. If the formula is right
the balance reaches (almost exactly) zero after n months.

In [ ]:
def simulate_balance(P, annual, months, payment=None):
    """Balance after each month, starting with P. Returns an array of length months + 1."""
    r = annual / 12.0
    pay = emi(P, annual, months) if payment is None else payment
    bal = [P]
    for _ in range(months):
        bal.append(bal[-1] * (1.0 + r) - pay)
    return np.array(bal)

## Step 4: the amortisation schedule
Month k: interest part = balance * r, principal part = EMI - interest part. As the balance
falls, the interest part falls and the principal part rises. Total interest = n EMI - P.

In [ ]:
def schedule(P, annual, months):
    """Return (interest, principal, balance_after) arrays, one entry per month."""
    r = annual / 12.0
    pay = emi(P, annual, months)
    interest = np.zeros(months)
    principal = np.zeros(months)
    balance = np.zeros(months)
    bal = P
    for k in range(months):
        interest[k] = bal * r
        principal[k] = pay - interest[k]
        bal -= principal[k]
        balance[k] = bal
    return interest, principal, balance


def total_interest(P, annual, months):
    return months * emi(P, annual, months) - P

## Step 5: tenure trade-off and the effective annual rate
A longer tenure lowers the EMI but raises the total interest. To compare offers quoted in
different ways use the effective annual rate  (1 + annual/12)^12 - 1.

In [ ]:
def tenure_table(P, annual, tenures):
    """List of (months, emi, total_interest) for each tenure."""
    return [(m, emi(P, annual, m), total_interest(P, annual, m)) for m in tenures]


def effective_annual_rate(annual, per_year=12):
    return (1.0 + annual / per_year) ** per_year - 1.0

## Step 6: a design problem
Two made-up offers for the same loan. Offer A has the lower headline rate but a processing
fee. Total cost = interest + fee. Which is cheaper?

In [ ]:
def total_cost(P, annual, months, fee_fraction=0.0):
    """Interest plus a one-off fee (fee_fraction of P). Money units."""
    return total_interest(P, annual, months) + fee_fraction * P

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    P, ann, m = 100000.0, 0.12, 24
    print(f"EMI for {P:.0f} at 12 percent over {m} months: {emi(P, ann, m):.2f}")
    print("final simulated balance:", round(simulate_balance(P, ann, m)[-1], 6))
    for n in (1, 2, 4, 12, 365, 100000):
        print(f"n = {n:6d}: {compound_growth(1000, 0.10, n, 5):.3f}")
    print(f"continuous limit: {continuous_growth(1000, 0.10, 5):.3f}")
    it, pr, bal = schedule(P, ann, m)
    print(f"month 1: interest {it[0]:.2f}, principal {pr[0]:.2f}; month {m}: interest {it[-1]:.2f}, principal {pr[-1]:.2f}")
    print(f"total interest {total_interest(P, ann, m):.2f}")
    for row in tenure_table(P, ann, (12, 24, 36, 48, 60)):
        print("tenure %2d months: EMI %9.2f, total interest %9.2f" % row)
    print(f"effective annual rate of 12 percent: {100 * effective_annual_rate(0.12):.3f} percent")
    for name, a, f in (("A", 0.090, 0.02), ("B", 0.098, 0.0)):
        print(f"offer {name}: EMI {emi(P, a, 36):.2f}, total cost {total_cost(P, a, 36, f):.2f}")

    months = np.arange(1, m + 1)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.bar(months, it, label="interest part")
    ax.bar(months, pr, bottom=it, label="principal part")
    ax.set_xlabel("month")
    ax.set_ylabel("EMI split (units)")
    ax.set_title("Each EMI: interest falls, principal rises")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "amortisation.png", dpi=120)

    ten = np.arange(6, 121, 6)
    tab = tenure_table(P, ann, ten)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    ax2.plot(ten, [x[1] for x in tab], "o-", color="C0", label="EMI (left)")
    ax2.set_xlabel("tenure (months)")
    ax2.set_ylabel("EMI (units per month)", color="C0")
    ax3 = ax2.twinx()
    ax3.plot(ten, [x[2] for x in tab], "s-", color="C3", label="total interest (right)")
    ax3.set_ylabel("total interest (units)", color="C3")
    ax2.set_title("Longer tenure: smaller EMI, much more interest")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "tenure_tradeoff.png", dpi=120)

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py` print `emi(100000, 0.12, 36)` and check by hand that it equals the 36-month row of the tenure table.
2. **Stretch.** Write `first_month_split(P, annual, months)` that returns the interest part and principal part of month 1, and test it on the worked example.
3. **Explain.** Explain why total interest equals n * EMI - P, and why it must also equal the sum of the interest parts in the schedule.

## Assessment
- **A. Conceptual.** Why is the interest part of the EMI large at the start and small at the end, although the EMI itself never changes?
- **B. Numerical.** Find the EMI and the total interest for 100000 units at 12 percent a year over 12 months. State the units and do a sanity check.
- **C. Graph and simulation interpretation.** In the tenure graph for 100000 units at 12 percent, what happens to the EMI and to the total interest when the tenure grows from 12 to 60 months?
- **D. Design and optimisation challenge.** Design: you can afford an EMI of at most 4000 on a 100000 loan at 12 percent and want the least total interest. Which tenure in the table (12, 24, 36, 48 or 60 months) do you pick? Separately, compare Offer A (9.0 percent a year with a 2 percent fee, 36 months) and Offer B (9.8 percent, no fee, 36 months) for 100000. Which is cheaper in total?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    P = 100000.0
    e = L.emi(P, 0.12, 24)
    checks.append(("worked example: EMI for 100000 at 12 percent over 24 months is 4707.35", abs(e - 4707.35) < 0.005))
    g = 1.01 ** 24
    checks.append(("hand calculation: 100000 * 0.01 * 1.01^24 / (1.01^24 - 1) matches emi()",
                   abs(P * 0.01 * g / (g - 1) - e) < 1e-6))
    bal = L.simulate_balance(P, 0.12, 24)
    checks.append(("simulating month by month, the balance hits zero after 24 payments", abs(bal[-1]) < 1e-6))
    checks.append(("the balance never rises and never goes below zero (to rounding)",
                   bool(np.all(np.diff(bal) < 0) and np.all(bal > -1e-6))))
    ok = True
    for ann, m in ((0.05, 12), (0.09, 60), (0.24, 120), (0.15, 7)):
        ok &= abs(L.simulate_balance(50000.0, ann, m)[-1]) < 1e-5
    checks.append(("simulation reaches zero for several other rates and tenures", bool(ok)))
    checks.append(("zero rate: EMI = P / n", abs(L.emi(120000.0, 0.0, 12) - 10000.0) < 1e-9))
    checks.append(("continuous limit: 1000 at 10 percent for 5 years tends to 1648.72",
                   abs(L.continuous_growth(1000, 0.10, 5) - 1648.721) < 1e-3))
    seq = [L.compound_growth(1000, 0.10, n, 5) for n in (1, 2, 4, 12, 365, 100000)]
    checks.append(("more compounding periods always give more, but never above the limit",
                   all(b > a for a, b in zip(seq, seq[1:])) and seq[-1] < L.continuous_growth(1000, 0.10, 5)
                   and abs(seq[-1] - L.continuous_growth(1000, 0.10, 5)) < 1e-3
                   and abs(seq[0] - 1610.51) < 1e-2 and abs(seq[3] - 1645.309) < 1e-3))
    it, pr, b = L.schedule(P, 0.12, 24)
    checks.append(("month 1: interest 1000.00 and principal 3707.35",
                   abs(it[0] - 1000.0) < 1e-6 and abs(pr[0] - 3707.35) < 0.005))
    checks.append(("month 24: interest 46.61 and principal 4660.74",
                   abs(it[-1] - 46.61) < 0.005 and abs(pr[-1] - 4660.74) < 0.005))
    checks.append(("interest part falls and principal part rises every month",
                   bool(np.all(np.diff(it) < 0) and np.all(np.diff(pr) > 0))))
    checks.append(("every month interest + principal = EMI", np.allclose(it + pr, e, atol=1e-9)))
    checks.append(("principal parts add up to P and the last balance is zero",
                   abs(pr.sum() - P) < 1e-6 and abs(b[-1]) < 1e-6))
    checks.append(("total interest = n EMI - P = sum of interest parts = 12976.33",
                   abs(L.total_interest(P, 0.12, 24) - 12976.33) < 0.005 and abs(it.sum() - L.total_interest(P, 0.12, 24)) < 1e-6))
    tab = L.tenure_table(P, 0.12, (12, 24, 36, 48, 60))
    checks.append(("tenure table: EMI 8884.88 (12 months) ... 2224.44 (60 months)",
                   abs(tab[0][1] - 8884.88) < 0.005 and abs(tab[-1][1] - 2224.44) < 0.005))
    checks.append(("tenure table: total interest 6618.55 (12 months) rises to 33466.69 (60 months)",
                   abs(tab[0][2] - 6618.55) < 0.005 and abs(tab[-1][2] - 33466.69) < 0.005))
    checks.append(("longer tenure: EMI always falls, total interest always rises",
                   all(a[1] > b2[1] and a[2] < b2[2] for a, b2 in zip(tab, tab[1:]))))
    checks.append(("5 times the tenure gives about a quarter of the EMI... but over 5 times the interest",
                   abs(tab[-1][1] / tab[0][1] - 0.25) < 0.001 and tab[-1][2] / tab[0][2] > 5.0))
    checks.append(("effective annual rate of 12 percent a year, paid monthly, is 12.683 percent",
                   abs(L.effective_annual_rate(0.12) - 0.126825) < 1e-6 and abs(L.effective_annual_rate(0.0)) < 1e-12))
    ca = L.total_cost(P, 0.090, 36, 0.02)
    cb = L.total_cost(P, 0.098, 36, 0.0)
    checks.append(("design: offer A (9.0 percent + 2 percent fee) costs 16479.04 in total",
                   abs(ca - 16479.04) < 0.005))
    checks.append(("design: offer B (9.8 percent, no fee) costs 15824.13 and is cheaper than A",
                   abs(cb - 15824.13) < 0.005 and cb < ca))
    checks.append(("offer A has the lower EMI (3179.97 against 3217.34) but the higher total cost",
                   abs(L.emi(P, 0.090, 36) - 3179.97) < 0.005 and abs(L.emi(P, 0.098, 36) - 3217.34) < 0.005))
    checks.append(("hand numbers: 1.01^24 = 1.2697 and 1.01^12 = 1.1268",
                   abs(1.01 ** 24 - 1.269735) < 1e-6 and abs(1.01 ** 12 - 1.126825) < 1e-6))
    checks.append(("tenure table: 36 months gives EMI 3321.43 and interest 19571.52; 24 months already gives EMI above 4000",
                   abs(tab[2][1] - 3321.43) < 0.005 and abs(tab[2][2] - 19571.52) < 0.005 and tab[1][1] > 4000))
    checks.append(("daily compounding (n = 365) gives 1648.61 for 1000 at 10 percent over 5 years",
                   abs(seq[4] - 1648.608) < 1e-3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
